In [10]:
import optuna
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np

In [3]:
# Load the Pima Indian Diabetes dataset (from UCI repository)
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']
df = pd.read_csv(url, names=columns)

In [4]:
# Replace zero values with NaN in columns where zero is not a valid value
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

# Impute the missing values with the mean of the respective column
df.fillna(df.mean(), inplace=True)

# Check if there are any remaining missing values
print(df.isnull().sum())

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [5]:
# Split into features (X) and target (y)
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Split data into training and test sets (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Optional: Scale the data for better model performance
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Check the shape of the data
print(f'Training set shape: {X_train.shape}')
print(f'Test set shape: {X_test.shape}')


Training set shape: (537, 8)
Test set shape: (231, 8)


# optimizing multiple models

In [7]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

In [11]:
# Define the objective function for Optuna
def objective(trial):
    # Choose the algorithm to tune
    classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest', 'GradientBoosting'])

    if classifier_name == 'SVM':
        # SVM hyperparameters
        c = trial.suggest_float('C', 0.1, 100, log=True)
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
        gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

        model = SVC(C=c, kernel=kernel, gamma=gamma, random_state=42)

    elif classifier_name == 'RandomForest':
        # Random Forest hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )

    elif classifier_name == 'GradientBoosting':
        # Gradient Boosting hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )

    # Perform cross-validation and return the mean accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score

In [12]:
# Create a study and optimize it using algorithm called TPEsampler(Tree-structured Parzen Estimator) - by default
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

[I 2025-12-03 18:59:25,910] A new study created in memory with name: no-name-1a3f26c9-3195-4b15-8d40-4887bd2a8d71
[I 2025-12-03 18:59:28,018] Trial 0 finished with value: 0.7653631284916201 and parameters: {'classifier': 'RandomForest', 'n_estimators': 194, 'max_depth': 19, 'min_samples_split': 6, 'min_samples_leaf': 1, 'bootstrap': True}. Best is trial 0 with value: 0.7653631284916201.
[I 2025-12-03 18:59:31,143] Trial 1 finished with value: 0.7504655493482307 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 169, 'learning_rate': 0.29982345781791575, 'max_depth': 9, 'min_samples_split': 7, 'min_samples_leaf': 5}. Best is trial 0 with value: 0.7653631284916201.
[I 2025-12-03 18:59:32,611] Trial 2 finished with value: 0.7188081936685289 and parameters: {'classifier': 'SVM', 'C': 6.342398322544789, 'kernel': 'poly', 'gamma': 'auto'}. Best is trial 0 with value: 0.7653631284916201.
[I 2025-12-03 18:59:34,518] Trial 3 finished with value: 0.74487895716946 and parameters: 

In [13]:
# Retrieve the best trial
best_trial = study.best_trial
print("Best trial parameters:", best_trial.params)
print("Best trial accuracy:", best_trial.value)

Best trial parameters: {'classifier': 'SVM', 'C': 0.11056829539223254, 'kernel': 'linear', 'gamma': 'scale'}
Best trial accuracy: 0.7895716945996275


In [14]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_C,params_bootstrap,params_classifier,params_gamma,params_kernel,params_learning_rate,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_n_estimators,state
0,0,0.765363,2025-12-03 18:59:25.912504,2025-12-03 18:59:28.018505,0 days 00:00:02.106001,NaN,True,RandomForest,NaN,NaN,NaN,19.0,1.0,6.0,194.0,COMPLETE
1,1,0.750466,2025-12-03 18:59:28.019813,2025-12-03 18:59:31.143001,0 days 00:00:03.123188,NaN,NaN,GradientBoosting,NaN,NaN,0.299823,9.0,5.0,7.0,169.0,COMPLETE
2,2,0.718808,2025-12-03 18:59:31.144303,2025-12-03 18:59:32.611294,0 days 00:00:01.466991,6.342398,NaN,SVM,auto,poly,NaN,NaN,NaN,NaN,NaN,COMPLETE
3,3,0.744879,2025-12-03 18:59:32.612728,2025-12-03 18:59:34.518708,0 days 00:00:01.905980,NaN,NaN,GradientBoosting,NaN,NaN,0.248505,4.0,8.0,8.0,203.0,COMPLETE
4,4,0.770950,2025-12-03 18:59:34.519940,2025-12-03 18:59:36.016337,0 days 00:00:01.496397,NaN,False,RandomForest,NaN,NaN,NaN,9.0,5.0,6.0,194.0,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.750466,2025-12-03 19:00:31.197922,2025-12-03 19:00:31.264996,0 days 00:00:00.067074,0.139593,NaN,SVM,scale,rbf,NaN,NaN,NaN,NaN,NaN,COMPLETE
96,96,0.752328,2025-12-03 19:00:31.266199,2025-12-03 19:00:32.755183,0 days 00:00:01.488984,NaN,False,RandomForest,NaN,NaN,NaN,3.0,4.0,9.0,233.0,COMPLETE
97,97,0.711359,2025-12-03 19:00:32.756476,2025-12-03 19:00:32.810793,0 days 00:00:00.054317,0.134741,NaN,SVM,scale,poly,NaN,NaN,NaN,NaN,NaN,COMPLETE
98,98,0.785847,2025-12-03 19:00:32.813275,2025-12-03 19:00:32.872387,0 days 00:00:00.059112,0.189452,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE


In [15]:
study.trials_dataframe()['params_classifier'].value_counts()

params_classifier
SVM                 77
GradientBoosting    12
RandomForest        11
Name: count, dtype: int64

In [ ]:
# most of the time svm is used

In [ ]:
# Define the objective function for Optuna
def objective(trial):
    # Choose the algorithm to tune
    classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest', 'GradientBoosting'])

    if classifier_name == 'SVM':
        # SVM hyperparameters
        # model obje

    elif classifier_name == 'RandomForest':
        # Random Forest hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )

    elif classifier_name == 'GradientBoosting':
        # Gradient Boosting hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )

    # Perform cross-validation and return the mean accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score